# 2. Ước lượng và kiểm định giả thuyết

Notebook cốt lõi cho Chương 4.1-4.5. Nội dung tập trung vào khoảng tin cậy và một số
kiểm định đại diện. Multiple-testing correction, bootstrap BCa và post-hoc đầy đủ được
giữ trong các notebook phân tích mở rộng 02-03.

In [ ]:
from pathlib import Path
import warnings

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

RANDOM_SEED = 42
ALPHA = 0.05
ROOT = Path("../..")
DATA_RAW = ROOT / "data" / "raw"
DATA_OUT = ROOT / "data" / "processed"
FIGURES = ROOT / "report" / "figures"
DATA_OUT.mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATA_OUT / "student_mat_clean.csv")

## 2.1. Khoảng tin cậy cho trung bình G3

In [ ]:
n = len(df)
mean_g3 = df["G3"].mean()
se = stats.sem(df["G3"])
ci_mean = stats.t.interval(1 - ALPHA, df=n - 1, loc=mean_g3, scale=se)
print(f"Mean G3 = {mean_g3:.3f}")
print(f"95% CI = [{ci_mean[0]:.3f}, {ci_mean[1]:.3f}]")

## 2.2. Hai nhóm: định hướng học đại học và G3

In [ ]:
yes = df.loc[df["higher"] == "yes", "G3"]
no = df.loc[df["higher"] == "no", "G3"]
t_stat, p_value = stats.ttest_ind(yes, no, equal_var=False)
diff = yes.mean() - no.mean()
se_diff = np.sqrt(yes.var(ddof=1)/len(yes) + no.var(ddof=1)/len(no))
df_welch = (yes.var(ddof=1)/len(yes) + no.var(ddof=1)/len(no))**2 / (
    (yes.var(ddof=1)/len(yes))**2/(len(yes)-1) + (no.var(ddof=1)/len(no))**2/(len(no)-1)
)
ci_diff = stats.t.interval(1 - ALPHA, df=df_welch, loc=diff, scale=se_diff)
print(f"Chênh lệch mean (yes - no) = {diff:.3f}")
print(f"Welch t={t_stat:.3f}, p={p_value:.4g}, 95% CI={ci_diff}")
print(f"Cỡ nhóm: yes={len(yes)}, no={len(no)}")

Nhóm `higher=no` chỉ có 20 học sinh. Chênh lệch quan sát được không phải tác động nhân
quả của việc có ý định học đại học, vì các nhóm không được phân bổ ngẫu nhiên.

## 2.3. Nhiều nhóm: số lần trượt môn trước đó

In [ ]:
failure_groups = [group["G3"].to_numpy() for _, group in df.groupby("failures")]
h_stat, p_failure = stats.kruskal(*failure_groups)
failure_summary = df.groupby("failures")["G3"].agg(["count", "mean", "median", "std"])
print(f"Kruskal-Wallis H={h_stat:.3f}, p={p_failure:.4g}")
failure_summary

## 2.4. Tương quan thứ hạng: Walc và G3

In [ ]:
rho, p_walc = stats.spearmanr(df["Walc"], df["G3"])
print(f"Spearman rho={rho:.3f}, p={p_walc:.4g}")

## 2.5. Bảng kết quả chính

In [ ]:
results = pd.DataFrame([
    {"phan_tich": "Mean G3", "uoc_luong": mean_g3, "ci_lower": ci_mean[0], "ci_upper": ci_mean[1], "p_value": np.nan},
    {"phan_tich": "higher: yes - no", "uoc_luong": diff, "ci_lower": ci_diff[0], "ci_upper": ci_diff[1], "p_value": p_value},
    {"phan_tich": "failures: Kruskal-Wallis", "uoc_luong": h_stat, "ci_lower": np.nan, "ci_upper": np.nan, "p_value": p_failure},
    {"phan_tich": "Walc vs G3: Spearman", "uoc_luong": rho, "ci_lower": np.nan, "ci_upper": np.nan, "p_value": p_walc},
])
results.to_csv(DATA_OUT / "course_inference_summary.csv", index=False)

fig, ax = plt.subplots(figsize=(7, 4))
sns.boxplot(data=df, x="failures", y="G3", ax=ax, color="#A0CBE8")
ax.set_title("G3 theo số lần trượt môn trước đó")
fig.tight_layout()
fig.savefig(FIGURES / "hyp_course_failures.png", dpi=150, bbox_inches="tight")
plt.show()
results

## Kết luận

- Khoảng tin cậy lượng hóa độ bất định thay vì chỉ báo cáo một giá trị trung bình.
- `higher` và `failures` có khác biệt đáng chú ý trong dữ liệu quan sát.
- Tương quan `Walc` với `G3` nhỏ và không còn thuyết phục khi xét toàn bộ họ giả thuyết
  trong phân tích mở rộng.
- Ý nghĩa thống kê không đồng nghĩa với ý nghĩa thực tiễn hoặc quan hệ nhân quả.